In [1]:
import pandas as pd, numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()
books = pd.read_csv("../data/books_with_emotions.csv")
embeddings = OpenAIEmbeddings()
db = Chroma(persist_directory="../data/chroma_db", embedding_function=embeddings)

In [7]:
# pick 3 random books
selected = books.sample(3)
picks = selected["isbn13"].tolist()
selected[["title", "authors"]]

,title,authors
4111,Writing with Intent,Margaret Atwood
301,Soul Mates,Thomas Moore
826,Pride and Prejudice,Jane Austen


In [8]:
descriptions = selected["description"].tolist()
vectors = embeddings.embed_documents(descriptions)
# create a new vector that averages the 3 books
avg = np.mean(vectors, axis=0).tolist()

In [14]:
# categories of the 3 entered books -- recommendations must be in this set
allowed_categories = set(selected["simple_categories"])
print("Allowed categories:", allowed_categories)

recs = db.similarity_search_by_vector(avg, k=50)   # pull more, filtering will thin it out
isbns = [int(r.page_content.strip('"').split()[0]) for r in recs]

result = books[books["isbn13"].isin(isbns)]
result = result[~result["isbn13"].isin(picks)]                          # drop the books you picked
result = result[result["simple_categories"].isin(allowed_categories)]   # keep only matching categories
result[["title", "authors", "simple_categories"]].head(10)

Allowed categories: {'Nonfiction', 'Fiction'}


,title,authors,simple_categories
298,The Complete Stories,Zora Neale Hurston,Fiction
300,The Infinite Plan,Isabel Allende,Fiction
404,Women,Charles Bukowski,Fiction
418,A Circle of Quiet,Madeleine L'Engle,Nonfiction
608,Collected Short Stories,Graham Greene,Fiction
637,Existentialists and Mystics,Iris Murdoch,Nonfiction
640,Wide Sargasso Sea,Jean Rhys,Nonfiction
786,A Room of One's Own,Virginia Woolf,Fiction
810,Seven Gothic Tales,Isak Dinesen,Fiction
828,The Turn of the Screw and The Aspern Papers,Henry James,Fiction


In [15]:
# Category-coherence test: pick 3 children's books, i want the recommendations to also be mostly
# children's books.

kids = books[books["simple_categories"].str.startswith("Children's")]
kids_trio = kids.sample(3, random_state=7)
kids_isbns = kids_trio["isbn13"].tolist()

print("Picked (children's):")
for t, c in zip(kids_trio["title"], kids_trio["simple_categories"]):
    print(f"  - {t}  [{c}]")

# average the 3 embeddings, search by that vector
kids_vec = np.mean(embeddings.embed_documents(kids_trio["description"].tolist()), axis=0).tolist()
kids_recs = db.similarity_search_by_vector(kids_vec, k=20)
kids_rec_isbns = [int(r.page_content.strip('"').split()[0]) for r in kids_recs]

rec = books[books["isbn13"].isin(kids_rec_isbns)]
rec = rec[~rec["isbn13"].isin(kids_isbns)].head(10)        # drop the picks

share = rec["simple_categories"].str.startswith("Children's").mean()
print(f"\nChildren's share of top 10 recommendations: {share:.0%}")
rec[["title", "authors", "simple_categories"]].reset_index(drop=True)

Picked (children's):
  - The Littles and Their Amazing New Friend  [Children's Fiction]
  - Harry Potter and the Half-Blood Prince (Book 6)  [Children's Fiction]
  - The Classic Treasury of Hans Christian Andersen  [Children's Fiction]

Children's share of top 10 recommendations: 50%


,title,authors,simple_categories
0,"The Lion, the Witch and the Wardrobe Read-Alou...",C. S. Lewis;Pauline Baynes,Children's Fiction
1,Fairy Tales,Terry Jones;Michael Foreman,Fiction
2,The Enchanted Castle,E. Nesbit,Children's Fiction
3,A Midsummer Night's Dream,William Shakespeare,Fiction
4,Westmark,Lloyd Alexander,Children's Fiction
5,The King of Elfland's Daughter,Edward John Moreton Drax Plunkett Baron Dunsany,Fiction
6,Switch on the Night,Ray Bradbury;Leo Dillon;Diane Dillon,Children's Fiction
7,A Wizard of Earthsea,Ursula K. Le Guin,Fiction
8,Reave the Just and Other Tales,Stephen R. Donaldson,Fiction
9,The witches,Roald Dahl,Children's Nonfiction
